# LeetCode #1146 — Snapshot Array (Mảng chụp trạng thái)

**Medium · Python 3 · Design · Binary Search · History per Index**  
[Đề gốc LeetCode](https://leetcode.com/problems/snapshot-array/) · [Đối chiếu mô tả và giới hạn](https://leetcode.doocs.org/en/lc/1146/)

```python
class SnapshotArray:
    def __init__(self, length: int): ...
    def set(self, index: int, val: int) -> None: ...
    def snap(self) -> int: ...
    def get(self, index: int, snap_id: int) -> int: ...
```

**Câu hỏi xuyên suốt:** Làm sao giữ lại những trạng thái cũ của mảng mà không cần sao chép **cả mảng** ở mỗi lần `snap()`?

## 1. Mục tiêu học tập

1. Phân biệt giá trị **đang có** với giá trị **đã lưu trong một snapshot**.
2. Hiểu `class`, `self`, `__init__`, thao tác `set/snap/get` trong LeetCode Design.
3. Tự xây cách baseline sao chép mảng: dễ hiểu nhưng tốn bộ nhớ.
4. Suy ra lưu **lịch sử `(snap_id, value)` của từng index**, tìm bằng binary search.
5. Nắm rõ cập nhật nhiều lần **trước cùng một snapshot**, phần tử chưa từng set, snapshots liên tiếp.
6. Có trace, code submit, đối chiếu ngẫu nhiên và 5 bài tập tự luyện.

## 2. Đề bài và ví dụ chuẩn

`SnapshotArray(length)` tạo mảng gồm `length` phần tử, **ban đầu đều bằng 0**.

- `set(index, val)`: đổi giá trị tại `index` trong **trạng thái hiện tại**.
- `snap()`: chụp trạng thái hiện tại và trả mã `snap_id` bắt đầu từ `0`.
- `get(index, snap_id)`: đọc **giá trị đã chụp** tại index, không phải giá trị hiện tại.

Ví dụ từ đề:

| Thao tác | Kết quả | Lý do |
|---|---|---|
| `SnapshotArray(3)` | `null` | Mảng `[0,0,0]` |
| `set(0,5)` | `null` | Hiện tại `[5,0,0]` |
| `snap()` | `0` | Lưu phiên bản 0 |
| `set(0,6)` | `null` | Sửa trạng thái *sau* snapshot 0 |
| `get(0,0)` | **`5`** | Snapshot 0 vẫn giữ `5` |

**Constraints:** `1<=length<=5*10^4`, `0<=val<=10^9`, tối đa `5*10^4` lời gọi `set/snap/get`; `get` dùng `snap_id` đã chụp hợp lệ.

## 3. Python nhập môn: mảng, chỉ số, gán phần tử

- `[0] * length` tạo list có `length` phần tử ban đầu bằng `0`.
- `arr[1] = 7`: đổi giá trị **đang có** ở index `1`.
- `arr.copy()` tạo list **khác**, không dùng chung nội dung list gốc.
- `=` giữa hai biến list **không** tự sao chép: `alias = arr` chỉ thêm một tên cho **cùng đối tượng**.

In [1]:
values=[0]*3
alias=values
frozen=values.copy()
values[0]=5
print('Mảng hiện tại:',values)
print('Alias cũng thay đổi:',alias)
print('Bản sao vẫn:',frozen)
assert alias[0]==5 and frozen[0]==0

Mảng hiện tại: [5, 0, 0]
Alias cũng thay đổi: [5, 0, 0]
Bản sao vẫn: [0, 0, 0]


## 4. Kiến thức về `class`, `self` và phương thức

`class` gom **dữ liệu** và **hành động** thành một đối tượng. `__init__` chạy khi gọi `Counter(5)`. `self.value` là thuộc tính *riêng từng đối tượng*. `self.inc()` gọi một phương thức khác của chính đối tượng ấy.

Vì đề #1146 yêu cầu dùng thiết kế lớp, hãy làm quen với cú pháp này trước.

In [2]:
class TinyCounter:
    def __init__(self, start=0):
        self.value=start
    def inc(self):
        self.value+=1
        return self.value

a=TinyCounter(5)
b=TinyCounter()
print('a:',a.inc(),'| b:',b.inc())
assert (a.value,b.value)==(6,1)

a: 6 | b: 1


## 5. Cách dễ hiểu nhất: lưu `arr.copy()` tại `snap()`

Ta giữ hai nơi:

- `current`: mảng đang sửa.
- `snapshots`: danh sách các **bản sao đầy đủ** đã chụp.

Khi `snap()`, thêm `current.copy()` vào `snapshots`, rồi trả index tương ứng. `get` chỉ cần `snapshots[snap_id][index]`.

**Ưu điểm:** rõ ràng, đúng logic. **Nhược điểm:** mỗi lần `snap()` sao chép `length` số, dù chỉ sửa một phần tử.

In [3]:
class SnapshotArrayNaive:
    def __init__(self,length):
        self.current=[0]*length
        self.snapshots=[]
    def set(self,index,val):
        self.current[index]=val
    def snap(self):
        self.snapshots.append(self.current.copy())
        return len(self.snapshots)-1
    def get(self,index,snap_id):
        return self.snapshots[snap_id][index]

naive=SnapshotArrayNaive(3)
naive.set(0,5)
first=naive.snap()
naive.set(0,6)
print('first =',first,'| get(0,0) =',naive.get(0,first),'| current =',naive.current)
assert (first,naive.get(0,0))==(0,5)

first = 0 | get(0,0) = 5 | current = [6, 0, 0]


## 6. Bài toán hiệu năng: nếu `length=50000`?

`50000` snapshots của mảng `50000` phần tử sẽ phải lưu tới **2,5 tỉ ô tham chiếu** theo kiểu sao chép mảng đầy đủ. Đây là ý tưởng **không phù hợp** giới hạn đề.

Cần đổi câu hỏi: ở mỗi index, giá trị **đổi khi nào**? Nếu không đổi, nó sẽ giữ nguyên từ snapshot trước.

Đây là cách nhìn **lưu các thay đổi (sparse history)** thay vì sao chép mọi trạng thái.

In [4]:
length=50_000
snaps=50_000
print('Số ô phải sao chép/lưu trong baseline tối đa:',length*snaps)
print('Nhưng nếu chỉ set 1 lần, lịch sử tối ưu có thể rất ngắn!')
assert length*snaps==2_500_000_000

Số ô phải sao chép/lưu trong baseline tối đa: 2500000000
Nhưng nếu chỉ set 1 lần, lịch sử tối ưu có thể rất ngắn!


## 7. Điểm đột phá: mỗi index có một lịch sử

Thay vì lưu toàn bộ snapshot, dùng cấu trúc:

```python
history = [
    [(0, 0)],              # index 0: ban đầu 0
    [(0, 0)],              # index 1: ban đầu 0
    [(0, 0)],              # index 2: ban đầu 0
]
```

Khi `set(0,5)` trong phiên bản 0, cập nhật lịch sử index 0 thành `[(0,5)]`.

Sau `snap()` trả 0, phiên bản **đang ghi** tăng lên 1. Nếu `set(0,6)`, thêm `(1,6)` ở cuối.

`get(0,0)` cần chọn **bản ghi cuối cùng có `snap_id <= 0`**, nên ra 5.

In [5]:
history_demo=[[(0,0)] for _ in range(3)]
history_demo[0][-1]=(0,5)
history_demo[0].append((1,6))
print('Lịch sử index 0:',history_demo[0])
print('Lịch sử index 1:',history_demo[1])
assert history_demo[0]==[(0,5),(1,6)]

Lịch sử index 0: [(0, 5), (1, 6)]
Lịch sử index 1: [(0, 0)]


## 8. Vì sao phải dùng list comprehension?

**Không** viết `history = [[(0,0)]] * length` nếu sẽ sửa list con, vì tất cả phần tử có thể tham chiếu cùng một list!

`history = [[(0,0)] for _ in range(length)]` tạo **mỗi list con độc lập**. Dấu `_` trong `for _ in ...` nghĩa là không cần giá trị vòng lặp.

In [6]:
wrong=[[(0,0)]]*3
wrong[0].append((1,7))
print('Sai do chia sẻ list con:', wrong)
right=[[(0,0)] for _ in range(3)]
right[0].append((1,7))
print('Đúng, lịch sử độc lập:', right)
assert len(wrong[1])==2 and len(right[1])==1

Sai do chia sẻ list con: [[(0, 0), (1, 7)], [(0, 0), (1, 7)], [(0, 0), (1, 7)]]
Đúng, lịch sử độc lập: [[(0, 0), (1, 7)], [(0, 0)], [(0, 0)]]


## 9. Mấu chốt của `set`: nhiều lần ghi trước cùng một snap

Giả sử đang ở `current_id=0`:

```text
set(0,5) → set(0,8) → snap() trả 0
```

Snapshot 0 chỉ cần giữ giá trị **cuối** `8`, không cần lưu cả `(0,5)` và `(0,8)`.

Quy tắc:

```python
if history[index][-1][0] == current_id:
    history[index][-1] = (current_id, val)   # ghi đè cùng phiên bản
else:
    history[index].append((current_id, val)) # phiên bản mới
```

Đây là **tối ưu bộ nhớ**, đồng thời bảo đảm `snap_id` trong mỗi lịch sử tăng **nghiêm ngặt**.

In [7]:
log=[(0,0)]
current_id=0
for val in [5,8,3]:
    if log[-1][0]==current_id:
        log[-1]=(current_id,val)
    else:
        log.append((current_id,val))
    print('set ->',val, '| history =',log)
assert log==[(0,3)]

set -> 5 | history = [(0, 5)]
set -> 8 | history = [(0, 8)]
set -> 3 | history = [(0, 3)]


## 10. Tìm bản ghi bằng Binary Search: tư duy trước code

Với `history[index]=[(0,5),(3,7),(8,9)]`, các truy vấn là:

| `snap_id` cần đọc | Chọn bản ghi cuối có id không vượt quá | Kết quả |
|---:|---|---:|
| `0` | `(0,5)` | 5 |
| `2` | `(0,5)` | 5 |
| `3` | `(3,7)` | 7 |
| `6` | `(3,7)` | 7 |
| `8` | `(8,9)` | 9 |

Các `snap_id` **đã được sắp tăng dần**, nên có thể tìm nhị phân thay vì đọc từng phần tử.

In [8]:
log=[(0,5),(3,7),(8,9)]
def linear_get(log,snap_id):
    ans=0
    for old_id,val in log:
        if old_id<=snap_id:
            ans=val
        else:
            break
    return ans

for snap_id in (0,2,3,6,8):
    print(snap_id,'->',linear_get(log,snap_id))
assert [linear_get(log,k) for k in (0,2,3,6,8)]==[5,5,7,7,9]

0 -> 5
2 -> 5
3 -> 7
6 -> 7
8 -> 9


## 11. `bisect_right` và vì sao cần `(snap_id, inf)`

`bisect_right(a, key)` trả vị trí **có thể chèn `key` vào sau các phần tử bằng nhau** trong list đã sắp. Tuples trong Python so sánh theo **thành phần thứ nhất, rồi thứ hai**.

Với history chứa các cặp `(snapshot_id, value)`, tìm:

```python
pos = bisect_right(history[index], (snap_id, float('inf'))) - 1
```

`float('inf')` lớn hơn mọi `val` hữu hạn, giúp bỏ qua **giá trị thứ hai** và chọn bản ghi mới nhất có id `<= snap_id`.

**Tránh nhầm:** `bisect_left(history, (snap_id, 0))` có thể chọn sai chính snapshot đang cần.

In [9]:
from bisect import bisect_right
example_log=[(0,5),(3,7),(8,9)]
for query_id in (0,2,3,6,8):
    pos=bisect_right(example_log,(query_id,float('inf')))-1
    print(f'get id={query_id}: pos={pos}, pair={example_log[pos]}')
    assert example_log[pos][1]==linear_get(example_log,query_id)

get id=0: pos=0, pair=(0, 5)
get id=2: pos=0, pair=(0, 5)
get id=3: pos=1, pair=(3, 7)
get id=6: pos=1, pair=(3, 7)
get id=8: pos=2, pair=(8, 9)


## 12. Ghép thành thiết kế hiệu quả

Mỗi vị trí bắt đầu bằng `[(0,0)]` để **chưa set lần nào vẫn đọc được 0**.

- `set`: ghi đè nếu thuộc current_id; nếu không thì append.
- `snap`: trả `current_id` rồi tăng lên 1. **Không chép mảng**.
- `get`: tìm bản ghi mới nhất không vượt quá `snap_id` bằng `bisect_right`.

In [10]:
class SnapshotArrayOptimized:
    def __init__(self,length):
        self.current_id=0
        self.history=[[(0,0)] for _ in range(length)]
    def set(self,index,val):
        log=self.history[index]
        if log[-1][0]==self.current_id:
            log[-1]=(self.current_id,val)
        else:
            log.append((self.current_id,val))
    def snap(self):
        snap_id=self.current_id
        self.current_id+=1
        return snap_id
    def get(self,index,snap_id):
        log=self.history[index]
        pos=bisect_right(log,(snap_id,float('inf')))-1
        return log[pos][1]

a=SnapshotArrayOptimized(3)
a.set(0,5)
snap0=a.snap()
a.set(0,6)
print('snap id:',snap0,'| get(0,0):',a.get(0,snap0))
print('History index0:',a.history[0])
assert (snap0,a.get(0,0))==(0,5)

snap id: 0 | get(0,0): 5
History index0: [(0, 5), (1, 6)]


## 13. Trace trạng thái quan trọng nhất: có / không có set

Giả sử `length=2`:

| Bước | Lệnh | `current_id` sau lệnh | `history[0]` | `history[1]` |
|---:|---|---:|---|---|
| 0 | khởi tạo | 0 | `[(0,0)]` | `[(0,0)]` |
| 1 | `set(0,5)` | 0 | `[(0,5)]` | `[(0,0)]` |
| 2 | `set(0,8)` | 0 | `[(0,8)]` | `[(0,0)]` |
| 3 | `snap()` trả 0 | 1 | `[(0,8)]` | `[(0,0)]` |
| 4 | `set(1,9)` | 1 | `[(0,8)]` | `[(0,0),(1,9)]` |
| 5 | `snap()` trả 1 | 2 | không đổi | không đổi |
| 6 | `get(1,0)` | 2 | không đổi | trả **0** |
| 7 | `get(1,1)` | 2 | không đổi | trả **9** |

In [11]:
def trace_snapshot_commands():
    a=SnapshotArrayOptimized(2)
    actions=[('set',0,5),('set',0,8),('snap',),('set',1,9),('snap',),('get',1,0),('get',1,1)]
    print('Khởi tạo: version=',a.current_id,'history=',a.history)
    for action in actions:
        method=action[0]
        out=getattr(a,method)(*action[1:])
        print(f'{action!r:>20}: result={out} | writing_id={a.current_id} | history={a.history}')

trace_snapshot_commands()

Khởi tạo: version= 0 history= [[(0, 0)], [(0, 0)]]
       ('set', 0, 5): result=None | writing_id=0 | history=[[(0, 5)], [(0, 0)]]
       ('set', 0, 8): result=None | writing_id=0 | history=[[(0, 8)], [(0, 0)]]
           ('snap',): result=0 | writing_id=1 | history=[[(0, 8)], [(0, 0)]]
       ('set', 1, 9): result=None | writing_id=1 | history=[[(0, 8)], [(0, 0), (1, 9)]]
           ('snap',): result=1 | writing_id=2 | history=[[(0, 8)], [(0, 0), (1, 9)]]
       ('get', 1, 0): result=0 | writing_id=2 | history=[[(0, 8)], [(0, 0), (1, 9)]]
       ('get', 1, 1): result=9 | writing_id=2 | history=[[(0, 8)], [(0, 0), (1, 9)]]


## 14. Code nộp LeetCode (đúng tên lớp và phương thức)

Đề thuộc dạng **Design**, không dùng `class Solution`. LeetCode sẽ tự khởi tạo `SnapshotArray`, gọi lần lượt `set`, `snap`, `get`.

Phần duy nhất cần ngoài lớp là `from bisect import bisect_right`.

In [12]:
from bisect import bisect_right

class SnapshotArray:
    def __init__(self, length: int):
        self.snap_id = 0
        self.history = [[(0, 0)] for _ in range(length)]

    def set(self, index: int, val: int) -> None:
        log = self.history[index]
        if log[-1][0] == self.snap_id:
            log[-1] = (self.snap_id, val)
        else:
            log.append((self.snap_id, val))

    def snap(self) -> int:
        result = self.snap_id
        self.snap_id += 1
        return result

    def get(self, index: int, snap_id: int) -> int:
        log = self.history[index]
        pos = bisect_right(log, (snap_id, float('inf'))) - 1
        return log[pos][1]

obj=SnapshotArray(3)
obj.set(0,5)
assert obj.snap()==0
obj.set(0,6)
assert obj.get(0,0)==5
print('Ví dụ chuẩn LeetCode: PASS')

Ví dụ chuẩn LeetCode: PASS


## 15. Các trường hợp biên hay bị sai

1. Gọi `snap()` **trước mọi set**: mảng chụp phải toàn `0`.
2. Gọi `set` nhiều lần trước cùng một `snap`: chỉ giá trị **cuối** được lưu.
3. Gọi `snap()` liên tiếp không `set`: mỗi lần trả một id mới, giá trị không đổi.
4. `set` sau snapshot **không sửa lịch sử** của snapshot cũ.
5. Giá trị có thể là `0` (trả về mặc định **và** set tường minh).
6. `get` với `snap_id` mới hơn lần cập nhật cuối ⇒ lấy giá trị cũ nhất còn hiệu lực.

In [13]:
b=SnapshotArray(2)
assert b.snap()==0
assert b.get(0,0)==0
b.set(1,4)
b.set(1,7)
assert b.snap()==1
assert b.get(1,0)==0 and b.get(1,1)==7
assert b.snap()==2
assert b.get(1,2)==7
b.set(1,0)
assert b.snap()==3
assert b.get(1,2)==7 and b.get(1,3)==0
assert b.get(0,3)==0
print('6 tình huống biên: PASS')

6 tình huống biên: PASS


## 16. Chứng minh tính đúng: bất biến của lịch sử

Sau mọi thao tác:

1. Mỗi `history[index]` có các mốc `snap_id` **tăng nghiêm ngặt** (do set cùng id sẽ ghi đè).
2. Bản ghi cuối cùng tại hoặc trước một snapshot `s` giữ đúng **giá trị cuối** được set tính tới lúc chụp snapshot `s`.
3. Nếu chưa có bản ghi set nào trước `s`, mốc gốc `(0,0)` giữ giá trị mặc định `0`.

`set` bảo toàn thứ tự và giá trị cuối; `snap` chỉ tăng id, không làm thay đổi snapshot cũ. `get` dùng binary search tìm **mốc lớn nhất không vượt quá `s`**, nên trả đúng giá trị đã chụp. Suy ra cả bốn phương thức đúng.

## 17. Độ phức tạp (phân biệt hai thiết kế)

Đặt `L` = độ dài mảng; `S` = số snapshot; `U` = tổng số lần **lưu bản ghi mới** do `set` (không tính các lần ghi đè cùng id); `k` = độ dài lịch sử của một index.

| Thao tác | Sao chép mảng (baseline) | Lưu lịch sử + binary search |
|---|---|---|
| Khởi tạo | `O(L)` | `O(L)` |
| `set` | `O(1)` | `O(1)` |
| `snap` | `O(L)` | **`O(1)`** |
| `get` | `O(1)` | **`O(log k)`** |
| Bộ nhớ | `O(L*S + L)` | **`O(L + U)`** |

Nếu có `S=0` thì tất nhiên chưa thể gọi `get`. Tổng `U` không vượt số lần set. Không được nói mọi `get` luôn `O(1)`.

## 18. Kiểm chứng ngẫu nhiên với baseline độc lập

Tạo cùng chuỗi lệnh hợp lệ, chạy **cả hai thiết kế**, so sánh kết quả của mọi `snap/get`. Phương pháp này giúp bắt lỗi dễ sót như `set` nhiều lần trong một phiên bản.

In [14]:
import random
rng=random.Random(1146)
checks=0
for rep in range(80):
    length=rng.randint(1,12)
    slow=SnapshotArrayNaive(length)
    fast=SnapshotArray(length)
    snaps=0
    for step in range(180):
        action=rng.choice(['set','set','set','snap','get'])
        if action=='set':
            i=rng.randrange(length)
            val=rng.randint(0,100)
            slow.set(i,val)
            fast.set(i,val)
        elif action=='snap':
            a=slow.snap()
            b=fast.snap()
            assert a==b
            snaps+=1
        elif snaps:
            i=rng.randrange(length)
            s=rng.randrange(snaps)
            assert fast.get(i,s)==slow.get(i,s),(rep,step,i,s)
            checks+=1
print('Đối chiếu', checks, 'truy vấn get ngẫu nhiên: PASS')

Đối chiếu 2903 truy vấn get ngẫu nhiên: PASS


## 19. Trace binary search thủ công (không dùng thư viện)

`lo, hi` là đoạn chỉ số **nửa mở `[lo,hi)`**. Ta cần tìm **vị trí đầu tiên có `record_id > snap_id`**, sau đó trừ 1.

Trong khi `lo < hi`:
- Nếu `log[mid][0] <= snap_id`, bỏ nửa trái tính tới `mid`, đặt `lo=mid+1`.
- Nếu lớn hơn, đặt `hi=mid`.

Lúc kết thúc, `lo-1` là bản ghi cần đọc. Vì luôn có bản ghi mặc định id 0 nên với `snap_id>=0` sẽ không bị âm.

In [15]:
def trace_history_binary_search(log, snap_id):
    lo,hi=0,len(log)
    while lo<hi:
        mid=(lo+hi)//2
        print(f'lo={lo}, hi={hi}, mid={mid}, record={log[mid]}, target={snap_id}')
        if log[mid][0] <= snap_id:
            lo=mid+1
        else:
            hi=mid
    pos=lo-1
    print('=> pos =',pos,'=> value =',log[pos][1])
    return log[pos][1]

assert trace_history_binary_search([(0,5),(3,7),(8,9)],6)==7

lo=0, hi=3, mid=1, record=(3, 7), target=6
lo=2, hi=3, mid=2, record=(8, 9), target=6
=> pos = 1 => value = 7


## 20. Các sai lầm phổ biến

| Lỗi | Điều xảy ra | Cách sửa |
|---|---|---|
| Lưu `self.current` trực tiếp khi snap | Các snapshot cũ bị thay đổi | Dùng `.copy()` ở baseline; lịch sử ở tối ưu |
| Sao chép toàn mảng khi snap | TLE/MLE với `50000` thao tác | Lưu bản ghi thay đổi theo index |
| Dùng `bisect_left(log,(snap_id,0))` | Có thể bỏ qua chính id đang cần | Dùng `bisect_right(log,(snap_id,inf))-1` |
| Tăng id **trước** khi trả `snap()` | Lần đầu trả `1` thay vì `0` | Lưu id cũ rồi tăng |
| Dùng `[[...]]*length` | Các index dùng chung list con | List comprehension |
| `set` nhiều lần cùng id cứ append | Tốn bộ nhớ, mất tăng nghiêm ngặt | Ghi đè bản ghi cuối |

In [16]:
from bisect import bisect_left
log=[(0,5),(3,7)]
bad=bisect_left(log,(3,0))-1
good=bisect_right(log,(3,float('inf')))-1
print('Sai ở snap_id=3:',log[bad][1],'| Đúng:',log[good][1])
assert log[bad][1]==5 and log[good][1]==7

Sai ở snap_id=3: 5 | Đúng: 7


## 21. Nhận dạng pattern

Bài có: **lịch sử nhiều phiên bản**, **truy vấn quá khứ**, **chỉ ít phần tử thay đổi giữa các phiên bản** ⇒ nghĩ tới lưu `(version,value)` cho từng key/index.

Liên hệ các kỹ thuật:
- **Sparse updates:** lưu thay đổi, không nhân bản dữ liệu.
- **Binary search on sorted timestamps:** truy vấn giá trị hiệu lực ở thời điểm `t`.
- **Versioning / temporal databases:** dữ liệu có thể thay đổi nhưng bản ghi lịch sử vẫn đọc được.

Không lạm dụng: nếu cần trả về toàn bộ mảng tại một snapshot, chi phí phải tính theo kích thước output nữa.

In [17]:
many=SnapshotArray(1000)
for _ in range(30):
    many.snap()     # 30 snapshot, không cần sao chép 1000 ô
stored_pairs=sum(len(h) for h in many.history)
print('30 snapshots, 1000 indexes: số cặp lịch sử thực sự lưu:',stored_pairs)
assert stored_pairs==1000

30 snapshots, 1000 indexes: số cặp lịch sử thực sự lưu: 1000


## 22. Bài tập — 5 câu

**Bài 1 — Dự đoán:** Khởi tạo `SnapshotArray(1)`, gọi `snap()` hai lần không có `set`. Hai id trả về và `get(0,1)` là gì?

**Bài 2 — Trace:** Thực hiện `set(0,2)`, `set(0,9)`, `snap()` (id0), `set(0,4)`, `snap()` (id1). Hãy viết `history[0]` tối ưu và hai kết quả get.

**Bài 3 — TODO:** viết `exercise_find(log, snap_id)` bằng **vòng lặp** quét lịch sử.

**Bài 4 — TODO:** viết `exercise_binary_search(log, snap_id)` bằng **binary search tự code**, không dùng `bisect`.

**Bài 5 — Thiết kế biến thể:** Chỉ từ `history`, **không thể luôn biết** phiên bản 0 đã có thao tác `set` hay chưa (vì `[(0,0)]` có thể là mặc định hoặc `set(0,0)`). Hãy tạo lớp kế thừa `SnapshotArray` ghi nhận thêm tập `snap_id` đã từng `set` tại từng index, rồi viết `count_versions_set(index)` trả số phiên bản khác nhau từng được set.

Các lời giải tham khảo nằm **phía sau phần gợi ý**.

In [18]:
# TODO — chỉ khai báo; test sau khi hoàn thiện, không gọi ngay.
def exercise_find(log, snap_id):
    # TODO: chọn giá trị tại bản ghi cuối có id <= snap_id
    pass

def exercise_binary_search(log, snap_id):
    # TODO: tìm vị trí đầu tiên có id > snap_id, trả phần tử trước đó
    pass

class ExerciseSnapshotArrayWithSetCount(SnapshotArray):
    # TODO: bổ sung self.touched_versions ở __init__.
    # TODO: override set(index,val), ghi current id rồi gọi super().set.
    # TODO: viết count_versions_set(index) trả số phiên bản đã được ghi.
    pass

# Sau khi làm, thử:
# assert exercise_find([(0,5),(3,7),(8,9)],6)==7
# assert exercise_binary_search([(0,5),(3,7),(8,9)],6)==7

## 23. Gợi ý 3 mức cho từng bài

| Bài | Mức 1 — ý tưởng | Mức 2 — công cụ | Mức 3 — sát code |
|---|---|---|---|
| 1 | Không set thì đều 0 | ID tăng từ 0 | Trả `0`, `1`, rồi `0` |
| 2 | Set cùng id giữ giá trị cuối | Cặp `(id,val)` | `[(0,9),(1,4)]` |
| 3 | Chạy log từ trái sang phải | `for old_id,val in log` | Khi `old_id<=snap_id` cập nhật `ans` |
| 4 | Tìm biên bên phải | `lo,hi,mid` | Nếu id ở `mid` <= target, `lo=mid+1` |
| 5 | Không thể suy ra lịch sử thao tác đầy đủ | Kế thừa class và `super()` | Lưu `set()` các id đã được set tại từng index, đếm `len` |

**Chú ý bài 5:** Đây là câu hỏi tư duy về giới hạn thông tin của mô hình hiện tại. Chỉ nhìn các cặp không luôn xác định được có `set` ở phiên bản 0 hay không.

## 24. Đáp án tham khảo — chỉ xem sau khi làm

**Bài 1:** `snap()` lần đầu trả `0`, lần hai trả `1`. Mảng chưa đổi nên `get(0,1)=0`.

**Bài 2:** `history[0]=[(0,9),(1,4)]`, lần lượt `get(0,0)=9`, `get(0,1)=4`.

**Bài 5:** Không thể biết chắc lịch sử thao tác qua `history`: `[(0,0)]` có thể là chưa set hoặc `set(0,0)`; `[(0,9)]` không cho biết đã set mấy lần trong id 0. Muốn đếm chính xác số **phiên bản có ít nhất một lần set**, cần metadata riêng. Đáp án dưới đây lưu một tập các id đã được cập nhật tại mỗi index.

In [19]:
def answer_find(log,snap_id):
    ans=0
    for old_id,val in log:
        if old_id>snap_id:
            break
        ans=val
    return ans

def answer_binary_search(log,snap_id):
    lo,hi=0,len(log)
    while lo<hi:
        mid=(lo+hi)//2
        if log[mid][0]<=snap_id:
            lo=mid+1
        else:
            hi=mid
    return log[lo-1][1]

class SnapshotArrayWithSetCount(SnapshotArray):
    # Biến thể để biết chính xác số phiên bản từng được set tại mỗi index.
    def __init__(self,length):
        super().__init__(length)
        self.touched_versions=[set() for _ in range(length)]
    def set(self,index,val):
        self.touched_versions[index].add(self.snap_id)
        super().set(index,val)
    def count_versions_set(self,index):
        return len(self.touched_versions[index])

for q,expect in [(0,5),(2,5),(3,7),(6,7),(8,9)]:
    log=[(0,5),(3,7),(8,9)]
    assert answer_find(log,q)==answer_binary_search(log,q)==expect
demo=SnapshotArrayWithSetCount(1)
demo.set(0,3)
demo.set(0,9)
demo.snap()
demo.set(0,9)
assert demo.count_versions_set(0)==2
print('Bài 3,4 và biến thể bài 5: PASS')

Bài 3,4 và biến thể bài 5: PASS


## 25. Tổng kết và checklist submit

```text
history[index] = [(0,0)]
set(index,val):
    nếu last_id == current_id -> ghi đè last
    ngược lại -> append((current_id,val))
snap():
    trả current_id; sau đó tăng current_id
get(index,id):
    rightmost pair có pair.id <= id  (binary search)
    trả pair.value
```

- [ ] Tên lớp **`SnapshotArray`**, không phải `Solution`.
- [ ] `snap()` lần đầu trả **0**; `get` không đọc trạng thái mới hơn.
- [ ] Giá trị mặc định là `0`; lịch sử các index độc lập.
- [ ] Không sao chép mảng ở `snap`; xử lý nhiều `set` trong cùng một phiên bản.
- [ ] `get` `O(log k)`, `set/snap` `O(1)`.
- [ ] Chạy ví dụ chuẩn và ca `snap` liên tục, `set` sau snap, `val=0`.